# Topic modelling for SEO with BERTopic (2026 refresh)

Companion notebook for the MLforSEO guide **"Topic Modelling for SEO with BERTopic"** by Lazarina Stoy.

**Part A: your pages (full text)**
1. Load your content (a crawl export) or the demo dataset
2. The simplest possible BERTopic run
3. "Build your own Lego": a custom pipeline (embeddings, UMAP, HDBSCAN, vectoriser, c-TF-IDF, representation model)
4. Inspect and visualise the topics
5. Fix the outlier bucket (topic -1) and rein in an over-produced topic list
6. More than one topic per page with `approximate_distribution`
7. Export

**Part B: short text (titles, reviews, queries)**
8. BERTopic vs LDA vs NMF on titles only, scored with the same coherence metric
9. Topic-general words: `reduce_frequent_words`
10. Human in the loop: seed words, merging and naming topics

Tested with **BERTopic 0.17.4**, sentence-transformers 6.2, Python 3.12. Runs in Google Colab; a GPU runtime makes the embedding step faster but is not required for a few thousand documents.

## 0. Install

In [ ]:
# In Colab, uncomment the next line (BERTopic and gensim are not preinstalled; gensim is only used for the coherence comparison)
# !pip install -q "bertopic>=0.17" "gensim>=4.4"

In [ ]:
import re, warnings
import numpy as np
import pandas as pd

import bertopic
from bertopic import BERTopic
warnings.filterwarnings("ignore", category=FutureWarning)
print("BERTopic", bertopic.__version__)
RANDOM_STATE = 42

## 1. Load your content

**Your own site:** export one row per page from your crawler (URL + main body text, and the title if you want Part B), save as CSV, set `USE_DEMO = False` and point the column names at it.

**Demo:** six groups from the public 20 Newsgroups dataset (cars, space, graphics, guns politics, hockey, medicine), around 3,400 user-written posts, each with a long body and a short subject line. It's the same demo as the LDA notebook, so you can compare the two directly.

In [ ]:
USE_DEMO = True

INPUT_CSV = "crawl_export.csv"
URL_COL   = "Address"
TEXT_COL  = "Body Text"
TITLE_COL = "Title 1"

if USE_DEMO:
    from sklearn.datasets import fetch_20newsgroups
    cats = ["rec.autos", "sci.space", "comp.graphics", "talk.politics.guns", "rec.sport.hockey", "sci.med"]
    raw = fetch_20newsgroups(subset="train", categories=cats, random_state=1)

    def split_post(post):
        head, _, body = post.partition("\n\n")
        m = re.search(r"^Subject:\s*(.*)$", head, flags=re.M)
        subject = re.sub(r"^(re:\s*)+", "", m.group(1).strip(), flags=re.I) if m else ""
        body = "\n".join(l for l in body.splitlines()
                         if not l.lstrip().startswith((">", "|")) and not re.search(r"writes:|wrote:", l))  # drop quoted replies
        return subject, body

    rows = [split_post(p) for p in raw.data]
    df = pd.DataFrame({"url": [f"demo-{i}" for i in range(len(rows))],
                       "title": [s for s, _ in rows], "text": [b for _, b in rows],
                       "label": [raw.target_names[t] for t in raw.target]})
else:
    src = pd.read_csv(INPUT_CSV)
    df = pd.DataFrame({"url": src[URL_COL], "title": src[TITLE_COL] if TITLE_COL in src else "",
                       "text": src[TEXT_COL].fillna("")})

df = df[df["text"].str.split().str.len() >= 20].reset_index(drop=True)

def light_clean(text):
    # BERTopic needs far less pre-processing than LDA: keep stopwords and sentence structure for the embedding model,
    # only strip what is pure noise (emails, URLs, excess whitespace)
    text = re.sub(r"\S+@\S+|https?://\S+|www\.\S+", " ", str(text))
    return re.sub(r"\s+", " ", text).strip()

docs = [light_clean(t) for t in df["text"]]
print(len(docs), "documents")

## 2. The simplest possible run
Genuinely all you need for a first pass.

In [ ]:
topic_model = BERTopic(language="english", min_topic_size=10)
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info().head(15)

## 3. Build your own Lego
Every block is optional and swappable. We compute the embeddings once and reuse them, so you can re-run the clustering with different settings without re-embedding.

* `ClassTfidfTransformer(reduce_frequent_words=True)` down-weights words that are frequent across *all* topics (the "topic-general words" problem).
* `KeyBERTInspired()` re-ranks each topic's words by how close they are to the topic's documents in embedding space, which usually gives cleaner labels.
* `random_state` on UMAP makes runs reproducible.

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")       # 384-dim embeddings, fast on CPU
embeddings = embedding_model.encode(docs, show_progress_bar=True)

umap_model    = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE)
hdbscan_model = HDBSCAN(min_cluster_size=30, min_samples=5, metric="euclidean", cluster_selection_method="eom", prediction_data=True)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
STOPWORDS_PLUS = list(ENGLISH_STOP_WORDS | {"article", "writes", "edu", "com"})   # add your brand and boilerplate words
vectorizer    = CountVectorizer(stop_words=STOPWORDS_PLUS, ngram_range=(1, 2), min_df=2)
ctfidf_model  = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(
    embedding_model      = embedding_model,
    umap_model           = umap_model,
    hdbscan_model        = hdbscan_model,
    vectorizer_model     = vectorizer,
    ctfidf_model         = ctfidf_model,
    representation_model = KeyBERTInspired(),
    calculate_probabilities = True,
    verbose = True,
)
topics, probs = topic_model.fit_transform(docs, embeddings)
info = topic_model.get_topic_info()
print(f"{len(info) - 1} topics, {(np.array(topics) == -1).mean():.0%} of documents in the outlier bucket (topic -1)")
info.head(15)

## 4. Inspect and visualise
Each `visualize_*` call returns an interactive Plotly figure. We also save them as HTML so you can share them.

In [ ]:
print(topic_model.get_topic(0)[:10])

In [ ]:
fig_topics = topic_model.visualize_topics();          fig_topics.write_html("bertopic_topic_map.html")
fig_bars   = topic_model.visualize_barchart(top_n_topics=12); fig_bars.write_html("bertopic_barchart.html")
fig_tree   = topic_model.visualize_hierarchy();       fig_tree.write_html("bertopic_hierarchy.html")
fig_bars

In [ ]:
if "label" in df:   # demo only: do the topics line up with the original groups?
    display(pd.crosstab(df["label"], pd.Series(topics, name="topic")).loc[:, lambda d: d.sum() > 25])

## 5. Outliers and over-production
HDBSCAN refuses to force a document into a cluster it doesn't fit, so some land in topic -1. That's honest, but for an internal-linking or content-audit project you usually want every page tagged. `reduce_outliers` reassigns them (here: to the nearest topic in embedding space), then `update_topics` refreshes the topic words.

If BERTopic gives you more topics than you can use, `reduce_topics` merges the most similar ones.

In [ ]:
new_topics = topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=embeddings)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer, ctfidf_model=ctfidf_model,
                          representation_model=KeyBERTInspired())
print(f"Outliers before: {(np.array(topics) == -1).mean():.0%}   after: {(np.array(new_topics) == -1).mean():.0%}")
topics = new_topics

In [ ]:
NR_TOPICS = 12        # set to None to keep everything
if NR_TOPICS and len(topic_model.get_topic_info()) - 1 > NR_TOPICS:
    topic_model.reduce_topics(docs, nr_topics=NR_TOPICS)
    topics = topic_model.topics_
topic_model.get_topic_info()

## 6. More than one topic per page
`fit_transform` gives each document one topic. Real pages cover several subtopics, which is LDA's strength. `approximate_distribution` closes much of that gap: it slides a window over each document, scores every chunk against every topic's c-TF-IDF representation, and returns a topic *distribution* per document.

Gotcha: chunks scoring below `min_similarity` (default 0.1) are ignored. With `reduce_frequent_words=True` the c-TF-IDF scores are small, so the default can leave almost every document with an all-zero distribution. Lowering it to around 0.02 fixes that (at 0 every chunk counts and the distributions go flat, so test a couple of values on your data).

In [ ]:
topic_distr, _ = topic_model.approximate_distribution(docs, window=8, stride=4, min_similarity=0.02, batch_size=500)
labels = {row.Topic: row.Name for row in topic_model.get_topic_info().itertuples() if row.Topic != -1}
distr = pd.DataFrame(topic_distr, columns=[labels[t] for t in sorted(labels)])
pages = pd.concat([df[["url", "title"]], distr.round(3)], axis=1)
pages["topic"] = topics
pages["topic_name"] = pages["topic"].map(labels)
pages["n_topics_over_20pct"] = (topic_distr > 0.2).sum(axis=1)
print(f"{(pages['n_topics_over_20pct'] > 1).mean():.0%} of documents have more than one topic above 20%")
pages.head()

In [ ]:
# visualise the topic mix of one document that spans several topics
doc_id = int(np.argmax((topic_distr > 0.1).sum(axis=1)))
print(df.at[doc_id, "title"])
topic_model.visualize_distribution(topic_distr[doc_id], min_probability=0.05)

## 7. Export
Every page tagged with its dominant topic, its topic mix, and the topic keywords: the artefact you take into an internal-linking or content-audit project.

In [ ]:
pages.to_csv("bertopic_page_topics.csv", index=False)
topic_model.get_topic_info().to_csv("bertopic_topic_info.csv", index=False)
print("Saved bertopic_page_topics.csv and bertopic_topic_info.csv")

# Part B: short text
Titles, headings, anchor text, reviews, queries, social posts: short text is where topic models struggle most (no context, data sparsity, noise, slang). Here we fit LDA, NMF and BERTopic on **titles only** and score all three with the same metric: c_v coherence of each model's top 10 words, measured against the full-text corpus so the numbers are comparable.

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.parsing.preprocessing import STOPWORDS
from gensim.utils import simple_preprocess
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

STOP = STOPWORDS.union({"edu", "com", "would", "like", "know", "think", "people", "get", "also", "use", "one", "writes", "article"})
def tokenise(text):
    return [t for t in simple_preprocess(light_clean(text), deacc=True, min_len=3) if t not in STOP and not t.isdigit()]

titles = df["title"].astype(str).tolist()
# replies in a thread share a subject line, so de-duplicate (you'd do the same with duplicate page titles)
seen, keep = set(), []
for i, t in enumerate(titles):
    key = t.lower().strip()
    if len(tokenise(t)) >= 2 and key not in seen:
        seen.add(key); keep.append(i)
short_docs = [titles[i] for i in keep]
short_tokens = [tokenise(t) for t in short_docs]
ref_tokens = [tokenise(df.at[i, "text"]) for i in keep]           # reference corpus for coherence
ref_dict = Dictionary(ref_tokens)
print(f"{len(short_docs)} titles, avg {np.mean([len(t) for t in short_tokens]):.1f} content words each")

def c_v(topic_word_lists):
    clean = [[w for w in ws if w in ref_dict.token2id][:10] for ws in topic_word_lists]
    clean = [ws for ws in clean if len(ws) >= 3]
    return CoherenceModel(topics=clean, texts=ref_tokens, dictionary=ref_dict, coherence="c_v", processes=1).get_coherence()

K = 8
# LDA on titles
d = Dictionary(short_tokens); d.filter_extremes(no_below=3, no_above=0.5)
lda_short = LdaModel([d.doc2bow(t) for t in short_tokens], id2word=d, num_topics=K, alpha="auto", eta="auto",
                     passes=10, iterations=200, random_state=RANDOM_STATE, eval_every=None)
lda_words = [[w for w, _ in lda_short.show_topic(t, topn=15)] for t in range(K)]
# NMF on titles
vec = TfidfVectorizer(analyzer=lambda x: x, min_df=3); X = vec.fit_transform(short_tokens)
nmf = NMF(n_components=K, init="nndsvda", random_state=RANDOM_STATE, max_iter=500).fit(X)
vocab = vec.get_feature_names_out()
nmf_words = [[vocab[i] for i in c.argsort()[::-1][:15]] for c in nmf.components_]

# BERTopic on titles (raw titles go in: the embedding model wants natural text)
short_emb = embedding_model.encode(short_docs, show_progress_bar=False)
bt_short = BERTopic(embedding_model=embedding_model,
                    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE),
                    hdbscan_model=HDBSCAN(min_cluster_size=15, prediction_data=True),
                    vectorizer_model=CountVectorizer(stop_words="english", min_df=2),
                    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True))
short_topics, _ = bt_short.fit_transform(short_docs, short_emb)
bt_words = [[w for w, _ in bt_short.get_topic(t)] for t in sorted(set(short_topics)) if t != -1]

comparison = pd.DataFrame([
    {"model": "LDA (titles)", "topics": K, "coherence_c_v": c_v(lda_words), "unassigned": "0%"},
    {"model": "NMF (titles)", "topics": K, "coherence_c_v": c_v(nmf_words), "unassigned": "0%"},
    {"model": "BERTopic (titles)", "topics": len(bt_words), "coherence_c_v": c_v(bt_words),
     "unassigned": f"{(np.array(short_topics) == -1).mean():.0%}"},
])
comparison.round(3)

In [ ]:
for name, words in [("LDA", lda_words), ("NMF", nmf_words), ("BERTopic", bt_words)]:
    print(f"\n{name}")
    for t, ws in enumerate(words[:8]):
        print(f"  {t}: {', '.join(ws[:8])}")

## 9. Topic-general words
Words that show up across every topic ("help", "question", "new", "need" in titles) blur topic labels. In LDA you remove them by hand or with frequency filters. In BERTopic, `reduce_frequent_words=True` in the c-TF-IDF step does it automatically. Compare the labels with and without it:

In [ ]:
plain = BERTopic(embedding_model=embedding_model,
                 umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE),
                 hdbscan_model=HDBSCAN(min_cluster_size=15, prediction_data=True),
                 vectorizer_model=CountVectorizer(min_df=2))        # no stopwords, no frequent-word reduction
plain.fit_transform(short_docs, short_emb)
reduced = BERTopic(embedding_model=embedding_model,
                   umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE),
                   hdbscan_model=HDBSCAN(min_cluster_size=15, prediction_data=True),
                   vectorizer_model=CountVectorizer(min_df=2),
                   ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True))
reduced.fit_transform(short_docs, short_emb)
pd.DataFrame({"without": plain.get_topic_info()["Name"].head(8).values,
              "with reduce_frequent_words": reduced.get_topic_info()["Name"].head(8).values})

## 10. Human in the loop
Topic models are exploratory, and your domain knowledge is part of the model. Three ways to put it in:

* **Seed words** (guided topic modelling): nudge the model towards topics you know matter.
* **Merge** topics that are really one theme.
* **Name** topics yourself, so the output speaks your business language.

In [ ]:
seed_topic_list = [["nasa", "space", "orbit", "launch"],
                   ["hockey", "nhl", "playoff", "team"],
                   ["gun", "firearms", "atf", "control"]]
guided = BERTopic(embedding_model=embedding_model, seed_topic_list=seed_topic_list,
                  umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE),
                  hdbscan_model=HDBSCAN(min_cluster_size=15, prediction_data=True),
                  vectorizer_model=CountVectorizer(stop_words="english", min_df=2),
                  ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True))
g_topics, _ = guided.fit_transform(short_docs, short_emb)
guided.get_topic_info().head(10)

In [ ]:
# merge two topics you consider one theme (pick the ids from the table above), then give topics your own labels
# guided.merge_topics(short_docs, [[1, 4]])
guided.set_topic_labels({t: f"Topic {t}: " + ", ".join(w for w, _ in guided.get_topic(t)[:3]) for t in guided.get_topics() if t != -1})
guided.get_topic_info()[["Topic", "Count", "CustomName"]].head(10)